# 07 Delay Propagation Score

## Objective
Combine structural centrality metrics with simulated failure impact to create a final station-level propagation risk score.

## Inputs
- `data/processed/station_metrics.csv`
- `data/processed/failure_results.csv`

## Outputs
- `data/processed/top_station_risk_scores.csv`
- `data/processed/top_station_summary.csv`

## Why this step matters
This notebook creates the final station ranking used in the dashboard. It moves beyond single-metric centrality and captures both structural importance and disruption severity.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

## Step 1: Merge network metrics with failure simulation outputs

This step combines centrality-based structural importance with simulated station failure impact.

In [3]:
metrics = pd.read_csv(PROCESSED_DIR / "station_metrics.csv")
failure_results = pd.read_csv(PROCESSED_DIR / "failure_results.csv")

print("metrics shape:", metrics.shape)
print("failure_results shape:", failure_results.shape)

metrics.head()
failure_results.head()

metrics shape: (272, 4)
failure_results shape: (272, 10)


,station_removed,num_components,largest_component_size,avg_shortest_path,components_norm,largest_component_loss,largest_component_loss_norm,avg_shortest_path_norm,failure_impact_score,station_display
0,camden town,3,251,13.667825,1.0,20,0.909091,0.093895,0.700896,Camden Town
1,earls court,3,262,14.673101,1.0,9,0.409091,0.354142,0.628970,Earls Court
2,finsbury park,3,259,13.870522,1.0,12,0.545455,0.146369,0.607547,Finsbury Park
3,stockwell,3,261,13.866578,1.0,10,0.454545,0.145349,0.579968,Stockwell
4,euston,2,249,13.731021,0.5,22,1.000000,0.110255,0.533077,Euston


In [4]:
risk_df = failure_results.merge(
    metrics,
    left_on="station_removed",
    right_on="station",
    how="left"
)

print("risk_df shape:", risk_df.shape)
risk_df.head()

risk_df shape: (272, 14)


,station_removed,num_components,largest_component_size,avg_shortest_path,components_norm,largest_component_loss,largest_component_loss_norm,avg_shortest_path_norm,failure_impact_score,station_display,station,degree_centrality,betweenness_centrality,closeness_centrality
0,camden town,3,251,13.667825,1.0,20,0.909091,0.093895,0.700896,Camden Town,camden town,0.01476,0.139921,0.095355
1,earls court,3,262,14.673101,1.0,9,0.409091,0.354142,0.628970,Earls Court,earls court,0.02214,0.205972,0.092904
2,finsbury park,3,259,13.870522,1.0,12,0.545455,0.146369,0.607547,Finsbury Park,finsbury park,0.01476,0.089354,0.089203
3,stockwell,3,261,13.866578,1.0,10,0.454545,0.145349,0.579968,Stockwell,stockwell,0.01476,0.074958,0.091678
4,euston,2,249,13.731021,0.5,22,1.000000,0.110255,0.533077,Euston,euston,0.01476,0.175398,0.103712


In [5]:
print(risk_df.isnull().sum())

station_removed                0
num_components                 0
largest_component_size         0
avg_shortest_path              0
components_norm                0
largest_component_loss         0
largest_component_loss_norm    0
avg_shortest_path_norm         0
failure_impact_score           0
station_display                0
station                        0
degree_centrality              0
betweenness_centrality         0
closeness_centrality           0
dtype: int64


## Step 2: Normalize centrality measures

Normalization places different metrics on a comparable scale before combining them into a composite score.

In [6]:
def minmax(series):
    denom = series.max() - series.min()
    if denom == 0:
        return pd.Series(0, index=series.index)
    return (series - series.min()) / denom

In [7]:
risk_df["betweenness_norm"] = minmax(risk_df["betweenness_centrality"])
risk_df["degree_norm"] = minmax(risk_df["degree_centrality"])
risk_df["closeness_norm"] = minmax(risk_df["closeness_centrality"])

## Step 3: Calculate the propagation risk score

The final score combines:
- betweenness centrality
- degree centrality
- failure impact score

In [8]:
risk_df["propagation_risk_score"] = (
    0.5 * risk_df["betweenness_norm"] +
    0.2 * risk_df["degree_norm"] +
    0.3 * risk_df["failure_impact_score"]
)

In [9]:
risk_df["station_display"] = (
    risk_df["station_removed"]
    .astype(str)
    .str.replace(" underground", "", regex=False)
    .str.title()
)

In [10]:
risk_df = risk_df.sort_values("propagation_risk_score", ascending=False)
risk_df.head(15)

,station_removed,num_components,largest_component_size,avg_shortest_path,components_norm,largest_component_loss,largest_component_loss_norm,avg_shortest_path_norm,failure_impact_score,station_display,station,degree_centrality,betweenness_centrality,closeness_centrality,betweenness_norm,degree_norm,closeness_norm,propagation_risk_score
70,baker street,1,271,17.167910,0.0,0,0.000000,1.000000,0.300000,Baker Street,baker street,0.02583,0.354381,0.114394,1.000000,1.000000,0.950907,0.790000
146,green park,1,271,14.429684,0.0,0,0.000000,0.291126,0.087338,Green Park,green park,0.02214,0.341879,0.118186,0.964722,0.833333,1.000000,0.675229
1,earls court,3,262,14.673101,1.0,9,0.409091,0.354142,0.628970,Earls Court,earls court,0.02214,0.205972,0.092904,0.581216,0.833333,0.672642,0.645965
7,kings cross st. pancras,2,254,14.127852,0.5,17,0.772727,0.212987,0.495714,Kings Cross St. Pancras,kings cross st. pancras,0.02583,0.195583,0.106108,0.551900,1.000000,0.843615,0.624664
141,waterloo,1,271,14.540823,0.0,0,0.000000,0.319897,0.095969,Waterloo,waterloo,0.02214,0.282364,0.108792,0.796781,0.833333,0.878363,0.593848
136,liverpool street,1,271,14.966407,0.0,0,0.000000,0.430073,0.129022,Liverpool Street,liverpool street,0.01845,0.272424,0.100259,0.768731,0.666667,0.767880,0.556405
8,stratford,2,251,13.407044,0.5,20,0.909091,0.026384,0.480642,Stratford,stratford,0.01107,0.222199,0.082723,0.627006,0.333333,0.540818,0.524362
13,wembley park,2,267,15.541015,0.5,4,0.181818,0.578828,0.428194,Wembley Park,wembley park,0.01845,0.182102,0.098260,0.513859,0.666667,0.741991,0.518721
143,bank,1,271,14.496078,0.0,0,0.000000,0.308314,0.092494,Bank,bank,0.01845,0.252756,0.105284,0.713232,0.666667,0.832940,0.517697
0,camden town,3,251,13.667825,1.0,20,0.909091,0.093895,0.700896,Camden Town,camden town,0.01476,0.139921,0.095355,0.394831,0.500000,0.704387,0.507684


## Step 4: Save final station risk rankings

This output is used directly in the Streamlit dashboard and final visualizations.

In [11]:
risk_df.to_csv(PROCESSED_DIR / "top_station_risk_scores.csv", index=False)
print("Saved top_station_risk_scores.csv")

Saved top_station_risk_scores.csv


In [12]:
top_station_summary = risk_df[[
    "station_removed",
    "station_display",
    "propagation_risk_score",
    "failure_impact_score",
    "betweenness_centrality",
    "degree_centrality"
]]

top_station_summary.to_csv(PROCESSED_DIR / "top_station_summary.csv", index=False)
print("Saved top_station_summary.csv")

Saved top_station_summary.csv
